In [ ]:

import random
from collections import defaultdict
!pip install deap
from deap import base, creator, tools

# ============================================================
# 16-Stone Game Using DEAP
# ============================================================
# Rules:
#   - Start with 16 stones
#   - Each player can take 1, 2, or 3 stones
#   - Player who takes the last stone wins
#
# AI:
#   - Chromosome length = 16
#   - Gene 0 means AI move when 1 stone is left
#   - Gene 1 means AI move when 2 stones are left
#   - ...
#   - Gene 15 means AI move when 16 stones are left
# ============================================================

STARTING_STONES = 16
MIN_TAKE = 1
MAX_TAKE = 3

# -----------------------------
# Evolution settings
# -----------------------------
POP_SIZE = 100
TOURNAMENT_SIZE = 3
CXPB = 0.8
MUTPB = 0.25
GENE_MUTATION_PROB = 0.12
ELITE_COUNT = 4

# -----------------------------
# Pre-training settings
# -----------------------------
PRETRAIN_TOTAL_GAMES = 1_000_000

# Each individual plays this many simulated games per generation
GAMES_PER_INDIVIDUAL_PRETRAIN = 50

# Total simulated games per generation = POP_SIZE * GAMES_PER_INDIVIDUAL_PRETRAIN
PRETRAIN_GENERATIONS = PRETRAIN_TOTAL_GAMES // (POP_SIZE * GAMES_PER_INDIVIDUAL_PRETRAIN)

# -----------------------------
# Human learning settings
# -----------------------------
GENERATIONS_AFTER_EACH_HUMAN_GAME = 40
GAMES_PER_INDIVIDUAL_HUMAN_LEARNING = 80

random.seed(42)


# ============================================================
# Helper functions
# ============================================================

def max_legal_take(stones):
    return min(MAX_TAKE, stones)


def legalize_move(move, stones):
    return max(MIN_TAKE, min(move, max_legal_take(stones)))


def random_legal_move(stones):
    return random.randint(MIN_TAKE, max_legal_take(stones))


def ai_move(individual, stones):
    """
    AI move from chromosome.
    individual[0] is move for 1 stone.
    individual[15] is move for 16 stones.
    """
    move = individual[stones - 1]
    return legalize_move(move, stones)


def print_strategy(individual):
    print("\nCurrent best AI strategy:")
    print("Stones left: ", end="")
    for s in range(1, STARTING_STONES + 1):
        print(f"{s:2d}", end=" ")
    print()

    print("AI takes   : ", end="")
    for s in range(1, STARTING_STONES + 1):
        print(f"{ai_move(individual, s):2d}", end=" ")
    print("\n")


def optimal_move_for_reference(stones):
    """
    This is not used by the AI.
    It is only used at the end to show comparison.
    """
    remainder = stones % 4

    if remainder == 0:
        return 1

    return remainder


def print_optimal_reference():
    print("\nReference optimal pattern:")
    print("Stones left: ", end="")
    for s in range(1, STARTING_STONES + 1):
        print(f"{s:2d}", end=" ")
    print()

    print("Best take  : ", end="")
    for s in range(1, STARTING_STONES + 1):
        print(f"{optimal_move_for_reference(s):2d}", end=" ")
    print("\n")


# ============================================================
# DEAP setup
# ============================================================

if not hasattr(creator, "FitnessMaxStone"):
    creator.create("FitnessMaxStone", base.Fitness, weights=(1.0,))

if not hasattr(creator, "StoneIndividual"):
    creator.create("StoneIndividual", list, fitness=creator.FitnessMaxStone)

toolbox = base.Toolbox()


def create_individual():
    """
    Create one AI strategy.
    Each gene is a move for a particular number of stones.
    """
    genes = []

    for stones in range(1, STARTING_STONES + 1):
        genes.append(random_legal_move(stones))

    return creator.StoneIndividual(genes)


toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxOnePoint)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)


# ============================================================
# Mutation
# ============================================================

def mutate_strategy(individual):
    """
    Mutate the AI strategy.

    Each gene has some probability of being replaced by a new legal move.
    """
    for i in range(len(individual)):
        stones = i + 1

        if random.random() < GENE_MUTATION_PROB:
            individual[i] = random_legal_move(stones)

    return (individual,)


toolbox.register("mutate", mutate_strategy)


# ============================================================
# Opponent models
# ============================================================

def random_opponent_move(stones):
    """
    Random opponent used during 1,000,000-game pre-training.
    """
    return random_legal_move(stones)


def predicted_human_move(stones, human_memory):
    """
    Opponent model after human games.

    If the human has previously played at this stone count,
    the simulated opponent often repeats one of those moves.

    Otherwise, it plays randomly.
    """
    if stones in human_memory and len(human_memory[stones]) > 0:
        if random.random() < 0.85:
            return legalize_move(random.choice(human_memory[stones]), stones)

    return random_legal_move(stones)


# ============================================================
# Game simulation
# ============================================================

def simulate_game(individual, ai_starts, opponent_mode, human_memory=None):
    """
    Simulates one full game.

    opponent_mode:
        "random" = opponent uses random moves
        "human_memory" = opponent imitates previous human moves
    """
    stones = STARTING_STONES
    ai_turn = ai_starts

    while stones > 0:

        if ai_turn:
            move = ai_move(individual, stones)
            stones -= move

            if stones == 0:
                return True

        else:
            if opponent_mode == "random":
                move = random_opponent_move(stones)

            elif opponent_mode == "human_memory":
                move = predicted_human_move(stones, human_memory)

            else:
                raise ValueError("Unknown opponent mode.")

            stones -= move

            if stones == 0:
                return False

        ai_turn = not ai_turn


def evaluate_individual(individual, opponent_mode, games_per_individual, human_memory=None):
    """
    Fitness function.

    Fitness = number of simulated games won by AI.
    """
    wins = 0

    for _ in range(games_per_individual):

        # Randomly allow AI or opponent to start
        ai_starts = random.choice([True, False])

        if simulate_game(
            individual=individual,
            ai_starts=ai_starts,
            opponent_mode=opponent_mode,
            human_memory=human_memory
        ):
            wins += 1

    return (wins,)


def invalidate_population_fitness(population):
    """
    Fitness must be recalculated whenever the training opponent changes.
    """
    for ind in population:
        if ind.fitness.valid:
            del ind.fitness.values


# ============================================================
# Evolution function
# ============================================================

def evolve_ai(
    population,
    generations,
    opponent_mode,
    games_per_individual,
    human_memory=None,
    title="Evolution"
):
    """
    Evolve AI population using DEAP.
    """

    invalidate_population_fitness(population)

    for gen in range(1, generations + 1):

        # Evaluate invalid individuals
        invalid_individuals = [ind for ind in population if not ind.fitness.valid]

        for ind in invalid_individuals:
            ind.fitness.values = evaluate_individual(
                individual=ind,
                opponent_mode=opponent_mode,
                games_per_individual=games_per_individual,
                human_memory=human_memory
            )

        # Sort population by fitness
        population.sort(key=lambda ind: ind.fitness.values[0], reverse=True)

        # Show progress
        if gen == 1 or gen % 20 == 0 or gen == generations:
            best = population[0]
            print(
                f"{title} | Generation {gen:4d}/{generations} "
                f"| Best fitness = {best.fitness.values[0]}"
            )

        # Elitism
        offspring = [toolbox.clone(ind) for ind in population[:ELITE_COUNT]]

        # Selection
        selected = toolbox.select(population, len(population) - ELITE_COUNT)
        selected = list(map(toolbox.clone, selected))

        # Crossover
        for child1, child2 in zip(selected[::2], selected[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        # Mutation
        for mutant in selected:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        offspring.extend(selected)

        population[:] = offspring

    # Final evaluation
    invalid_individuals = [ind for ind in population if not ind.fitness.valid]

    for ind in invalid_individuals:
        ind.fitness.values = evaluate_individual(
            individual=ind,
            opponent_mode=opponent_mode,
            games_per_individual=games_per_individual,
            human_memory=human_memory
        )

    population.sort(key=lambda ind: ind.fitness.values[0], reverse=True)

    best_individual = toolbox.clone(population[0])

    return population, best_individual


# ============================================================
# Human interaction
# ============================================================

def get_human_move(stones):
    while True:
        try:
            move = int(input(f"Your move. Stones left = {stones}. Take 1, 2, or 3: "))

            if MIN_TAKE <= move <= max_legal_take(stones):
                return move
            else:
                print(f"Invalid move. You can take from 1 to {max_legal_take(stones)}.")

        except ValueError:
            print("Please enter a number.")


def ask_who_starts():
    while True:
        choice = input("\nWho starts? Enter H for human or A for AI: ").strip().lower()

        if choice == "h":
            return False

        elif choice == "a":
            return True

        else:
            print("Please enter H or A.")


def play_human_game(best_ai, human_memory):
    """
    Play one real game against human.
    AI moves automatically.
    Human moves are stored for future learning.
    """
    stones = STARTING_STONES
    ai_starts = ask_who_starts()
    ai_turn = ai_starts

    print("\nNew game started.")
    print(f"Starting stones = {STARTING_STONES}")

    while stones > 0:

        if ai_turn:
            move = ai_move(best_ai, stones)
            stones -= move

            print(f"\nAI takes {move} stone(s).")
            print(f"Stones remaining = {stones}")

            if stones == 0:
                print("\nAI wins!")
                return "AI"

        else:
            state_before_move = stones
            move = get_human_move(stones)

            # Store human move for future evolution
            human_memory[state_before_move].append(move)

            stones -= move

            print(f"You took {move} stone(s).")
            print(f"Stones remaining = {stones}")

            if stones == 0:
                print("\nHuman wins!")
                return "Human"

        ai_turn = not ai_turn


def print_human_memory(human_memory):
    if len(human_memory) == 0:
        print("\nNo human moves stored yet.")
        return

    print("\nHuman move memory collected so far:")

    for stones in sorted(human_memory.keys(), reverse=True):
        print(f"When {stones:2d} stones were left, human took: {human_memory[stones]}")


# ============================================================
# Main program
# ============================================================

def main():
    print("============================================")
    print(" DEAP Evolutionary AI for 16-Stone Game")
    print("============================================")

    print("\nRules:")
    print("1. Start with 16 stones.")
    print("2. Each player takes 1, 2, or 3 stones.")
    print("3. The player who takes the last stone wins.")

    print("\nAI representation:")
    print("Chromosome length = 16")
    print("Each gene tells the AI what to take for a given number of stones.")

    print("\nCreating initial random population...")
    population = toolbox.population(n=POP_SIZE)

    human_memory = defaultdict(list)

    print("\n============================================")
    print(" Pre-training phase")
    print("============================================")
    print(f"AI will now play approximately {PRETRAIN_TOTAL_GAMES:,} simulated games.")
    print("Opponent moves are random during this phase.")
    print(f"Population size = {POP_SIZE}")
    print(f"Games per individual per generation = {GAMES_PER_INDIVIDUAL_PRETRAIN}")
    print(f"Pre-training generations = {PRETRAIN_GENERATIONS}")
    print(
        f"Total simulated games = "
        f"{POP_SIZE * GAMES_PER_INDIVIDUAL_PRETRAIN * PRETRAIN_GENERATIONS:,}"
    )

    population, best_ai = evolve_ai(
        population=population,
        generations=PRETRAIN_GENERATIONS,
        opponent_mode="random",
        games_per_individual=GAMES_PER_INDIVIDUAL_PRETRAIN,
        human_memory=None,
        title="Pre-training"
    )

    print("\nPre-training completed.")
    print_strategy(best_ai)
    print_optimal_reference()

    game_number = 1

    while True:
        print(f"\n========== Human Game {game_number} ==========")

        print("\nAI will play using its current best evolved strategy.")
        print_strategy(best_ai)

        winner = play_human_game(best_ai, human_memory)

        print(f"\nWinner of this game: {winner}")

        print_human_memory(human_memory)

        print("\n============================================")
        print(" Human-based learning phase")
        print("============================================")
        print("AI will now evolve using your previous moves.")

        population, best_ai = evolve_ai(
            population=population,
            generations=GENERATIONS_AFTER_EACH_HUMAN_GAME,
            opponent_mode="human_memory",
            games_per_individual=GAMES_PER_INDIVIDUAL_HUMAN_LEARNING,
            human_memory=human_memory,
            title="Human learning"
        )

        print("\nAI learning after human game completed.")
        print_strategy(best_ai)

        again = input("Play again? Enter Y for yes or N for no: ").strip().lower()

        if again != "y":
            print("\nGame finished.")
            break

        game_number += 1


if __name__ == "__main__":
    main()

 DEAP Evolutionary AI for 16-Stone Game

Rules:
1. Start with 16 stones.
2. Each player takes 1, 2, or 3 stones.
3. The player who takes the last stone wins.

AI representation:
Chromosome length = 16
Each gene tells the AI what to take for a given number of stones.

Creating initial random population...

 Pre-training phase
AI will now play approximately 1,000,000 simulated games.
Opponent moves are random during this phase.
Population size = 100
Games per individual per generation = 50
Pre-training generations = 200
Total simulated games = 1,000,000
Pre-training | Generation    1/200 | Best fitness = 44.0
Pre-training | Generation   20/200 | Best fitness = 50.0
Pre-training | Generation   40/200 | Best fitness = 50.0
Pre-training | Generation   60/200 | Best fitness = 50.0
Pre-training | Generation   80/200 | Best fitness = 50.0
Pre-training | Generation  100/200 | Best fitness = 50.0
Pre-training | Generation  120/200 | Best fitness = 50.0
Pre-training | Generation  140/200 | Best f

You took 1 stone(s).
Stones remaining = 3

AI takes 3 stone(s).
Stones remaining = 0

AI wins!

Winner of this game: AI

Human move memory collected so far:
When 16 stones were left, human took: [2, 1, 1, 1]
When 13 stones were left, human took: [1, 1]
When 12 stones were left, human took: [2, 2]
When  9 stones were left, human took: [1, 1]
When  8 stones were left, human took: [1, 3]
When  6 stones were left, human took: [2]
When  5 stones were left, human took: [1]
When  4 stones were left, human took: [3, 1]
When  3 stones were left, human took: [3, 3]

 Human-based learning phase
AI will now evolve using your previous moves.
Human learning | Generation    1/40 | Best fitness = 80.0
Human learning | Generation   20/40 | Best fitness = 80.0
